# Suggested chunk shapes for ECCOFS

Opens the virtual Icechunk on source.coop and uses `dynamic_chunks.iterative_ratio_increase_algo` to suggest ~10 MB chunk shapes.
The **aspect ratio** is the desired relative number of chunks along (time, level, y, x); `-1` means "don't chunk this dimension".
Only shape and dtype matter, so nothing is read from the source apart from metadata.

Run with `dynamic_chunks` on the path, e.g. `PYTHONPATH=/shared/code-sprint-2026/dynamic_chunks`.

In [ ]:
import matplotlib.pyplot as plt
import zarr

from rechunk_vars import open_source
from suggest_chunks import RATIOS, compare, suggest, summarize

zarr.config.set({"async.concurrency": 32})
ds = open_source()
ds[["temp", "u", "v"]]

## Variable to inspect

In [ ]:
var = "u"   # temp, u or v
da = ds[var]
print(f"{var}: dims={da.dims} shape={da.shape} dtype={da.dtype} total={da.nbytes/1e9:.0f} GB")

## Current chunking vs. suggestions

The `RATIOS` presets are in `suggest_chunks.py`. Add your own to compare.

In [ ]:
ratios = {**RATIOS}   # e.g. ratios["time series"] = (-1, 1, 4, 4)
table = compare(da, ratios, target="10MB", tolerance=0.2)
table.style.format({"chunk MB": "{:.2f}"})

## What each layout looks like

Bars show chunk length as a fraction of the full dimension (full bar = dimension not chunked).

In [ ]:
fig, axes = plt.subplots(1, len(table), figsize=(3.2 * len(table), 3), sharey=True)
for ax, (name, row) in zip(axes, table.iterrows()):
    frac = [c / n for c, n in zip(row["chunks"], da.shape)]
    ax.barh(range(4), frac, color="tab:blue")
    ax.set_xscale("log"); ax.set_xlim(1e-3, 6)
    ax.set_yticks(range(4)); ax.set_yticklabels(da.dims); ax.invert_yaxis()
    for i, c in enumerate(row["chunks"]):
        ax.text(frac[i] * 1.15, i, str(c), va="center", fontsize=9)
    ax.set_title(f"{name}\n{row['chunk MB']:.1f} MB, {row['total chunks']:,} chunks", fontsize=10)
    ax.set_xlabel("chunk / dimension")
plt.tight_layout()

## Try your own ratio

In [ ]:
ratio = (-1, 1, 4, 4)   # (time, level, y, x)
try:
    chunks = suggest(da, ratio, target="10MB", tolerance=0.3)
    print(summarize(da, chunks))
except Exception as e:
    print("no match within tolerance:", e)